# TUGAS MINGGU 11 — IMPLEMENTASI ELT PIPELINE
**Mata Kuliah:** Big Data  
**Kelompok 8**
- Alif Akbar Arraiyan — 101032330121
- Farid Yusron Syabani — 101032300192

## Ringkasan
Notebook ini mengimplementasikan pipeline **ELT (Extract – Load – Transform)** untuk studi kasus
**Analisis Kecelakaan Lalu Lintas Amerika Serikat (2020–2024)**, mengacu langsung pada rancangan
arsitektur ELT Minggu 9: *Airbyte (Extract & Load raw JSONB) → Supabase PostgreSQL (skema `raw` →
transformasi SQL oleh dbt ke skema `analytics`) → Power BI*, dengan **Apache Airflow** sebagai orkestrator.

Berbeda dari pendekatan **ETL** pada Minggu 10 (*transformasi di RAM dengan pandas sebelum disimpan*),
pendekatan **ELT** pada Minggu 11:
1. **Extract**: mengambil data mentah dari sumber tanpa pembersihan maupun konversi tipe apa pun.
2. **Load (Raw Data)**: memuat seluruh data mentah dalam format JSON apa adanya ke skema **`raw`**
   (`raw_accidents`, `raw_nhtsa_api_cases`, `raw_nhtsa_states`) pada target storage.
3. **Transform (In-Database SQL)**: menjalankan *data inspection*, *data cleaning*, *feature engineering*,
   *standardisasi/encoding*, dan *JOIN* **langsung di dalam engine database** melalui query SQL
   (`json_extract`, CTE, window function `ROW_NUMBER()`) untuk menghasilkan skema **`analytics`**
   (`analytics_accidents`, `analytics_nhtsa_api_cases`), sementara tabel `raw_*` tetap dipertahankan terpisah.

Implementasi lokal memakai **SQLite 3.53.1** (`supabase_local.db`) sebagai representasi terstruktur
skema `raw` dan `analytics` di Supabase PostgreSQL. Fungsi `run_supabase_pg_elt()` dan skrip
`postgres_elt_transform.sql` tersedia untuk eksekusi langsung pada instance Supabase PostgreSQL cloud.

Urutan eksekusi: **Extract → Load (Raw) → Transform (In-DB SQL) → Integrasi → Testing**.


In [1]:
import os, sys, csv, json, time, sqlite3, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np


def find_project_root(start=Path.cwd()):
    p = Path(start).resolve()
    for cand in [p] + list(p.parents):
        if (cand / "data" / "processed").is_dir():
            return cand
    raise FileNotFoundError("Root repository tidak ditemukan (folder data/processed tidak ada)")


PROJECT_ROOT = find_project_root()
SOURCE_CSV   = PROJECT_ROOT / "data" / "processed" / "US_Accidents_Cleaned_NHTSA_2020-2024.csv"
API_SNAPSHOT = PROJECT_ROOT / "data" / "raw" / "nhtsa_api_live.json"
API_ENDPOINT = ("https://crashviewer.nhtsa.dot.gov/crashviewer/CrashAPI/crashes/GetCaseList"
                "?states=1,51&fromYear={year}&toYear={year}"
                "&minNumOfVehicles=1&maxNumOfVehicles=6&format=json")

DELIVERABLE  = PROJECT_ROOT / "tugas" / "minggu11" / "Tugas_Minggu11_Kelompok8"
for sub in ["Source_Code", "Dataset", "Documentation", "Screenshots"]:
    (DELIVERABLE / sub).mkdir(parents=True, exist_ok=True)

RAW_MAIN_CSV     = DELIVERABLE / "Dataset" / "raw_dataset.csv"
RAW_API_CSV      = DELIVERABLE / "Dataset" / "raw_nhtsa_api_cases.csv"
TRANSFORMED_CSV  = DELIVERABLE / "Dataset" / "transformed_dataset.csv"
API_CASES_CSV    = DELIVERABLE / "Dataset" / "nhtsa_api_cases.csv"
WAREHOUSE_DB     = DELIVERABLE / "Dataset" / "supabase_local.db"
METRICS_JSON     = DELIVERABLE / "Documentation" / "elt_metrics.json"
LOG_FILE         = DELIVERABLE / "Screenshots" / "pipeline_log.txt"
PG_SQL_FILE      = DELIVERABLE / "Source_Code" / "postgres_elt_transform.sql"

LOGS = []


def log(msg, level="INFO"):
    line = f"[{datetime.now():%H:%M:%S}] [{level}] {msg}"
    LOGS.append(line)
    print(line, flush=True)


# Referensi otoritatif NHTSA CrashAPI definitions/GetStates (53 entitas)
NHTSA_STATES = {
    "AK": (2, "Alaska", "West"), "AL": (1, "Alabama", "South"), "AR": (5, "Arkansas", "South"),
    "AZ": (4, "Arizona", "West"), "CA": (6, "California", "West"), "CO": (8, "Colorado", "West"),
    "CT": (9, "Connecticut", "Northeast"), "DC": (11, "District of Columbia", "South"),
    "DE": (10, "Delaware", "South"), "FL": (12, "Florida", "South"), "GA": (13, "Georgia", "South"),
    "HI": (15, "Hawaii", "West"), "IA": (19, "Iowa", "Midwest"), "ID": (16, "Idaho", "West"),
    "IL": (17, "Illinois", "Midwest"), "IN": (18, "Indiana", "Midwest"), "KS": (20, "Kansas", "Midwest"),
    "KY": (21, "Kentucky", "South"), "LA": (22, "Louisiana", "South"), "MA": (25, "Massachusetts", "Northeast"),
    "MD": (24, "Maryland", "South"), "ME": (23, "Maine", "Northeast"), "MI": (26, "Michigan", "Midwest"),
    "MN": (27, "Minnesota", "Midwest"), "MO": (29, "Missouri", "Midwest"), "MS": (28, "Mississippi", "South"),
    "MT": (30, "Montana", "West"), "NC": (37, "North Carolina", "South"), "ND": (38, "North Dakota", "Midwest"),
    "NE": (31, "Nebraska", "Midwest"), "NH": (33, "New Hampshire", "Northeast"),
    "NJ": (34, "New Jersey", "Northeast"), "NM": (35, "New Mexico", "West"), "NV": (32, "Nevada", "West"),
    "NY": (36, "New York", "Northeast"), "OH": (39, "Ohio", "Midwest"), "OK": (40, "Oklahoma", "South"),
    "OR": (41, "Oregon", "West"), "PA": (42, "Pennsylvania", "Northeast"), "PR": (43, "Puerto Rico", "South"),
    "RI": (44, "Rhode Island", "Northeast"), "SC": (45, "South Carolina", "South"),
    "SD": (46, "South Dakota", "Midwest"), "TN": (47, "Tennessee", "South"), "TX": (48, "Texas", "South"),
    "UT": (49, "Utah", "West"), "VA": (51, "Virginia", "South"), "VI": (52, "Virgin Islands", "South"),
    "VT": (50, "Vermont", "Northeast"), "WA": (53, "Washington", "West"), "WI": (55, "Wisconsin", "Midwest"),
    "WV": (54, "West Virginia", "South"), "WY": (56, "Wyoming", "West"),
}


def open_sqlite_elt_conn(db_path):
    """Buka koneksi SQLite dan daftarkan fungsi SQL INITCAP() agar setara PostgreSQL."""
    conn = sqlite3.connect(db_path)
    conn.create_function("INITCAP", 1, lambda s: str(s).title() if s is not None else None)
    return conn


log("Konfigurasi ELT siap | root: " + str(PROJECT_ROOT))


[14:51:17] [INFO] Konfigurasi ELT siap | root: /home/relyf/Big Data


## 1. Extract — Data Extraction (Raw, Tanpa Transformasi)

**Sumber data (sama dengan Minggu 9 & 10):**
1. Dataset historis konsolidasi Kaggle (`data/processed/US_Accidents_Cleaned_NHTSA_2020-2024.csv`) — **266.656 baris × 21 kolom**.
2. Snapshot **NHTSA Crash API** (`data/raw/nhtsa_api_live.json`, endpoint `GetCaseList` states `1,51`, tahun `2020–2026`) — **8.798 kasus fatal mentah × 9 atribut**.
3. Referensi otoritatif **NHTSA State Definitions** (`definitions/GetStates`, cached lokal) — **53 entitas state/region**.

**Prinsip ELT pada tahap Extract:**
- Seluruh data dibaca dalam kondisi **mentah apa adanya** (`dtype=str` pada CSV dan objek JSON asli pada API) **tanpa cleaning, tanpa parsing tanggal, tanpa penghapusan duplikat, dan tanpa konversi tipe**.
- Pemeriksaan awal hanya mencatat dimensi, tipe atribut awal, contoh baris, serta indikasi nilai kosong/duplikat/inkonsistensi format sebelum dimuat ke `raw` storage.


In [2]:
def fetch_live_api(year):
    """Cadangan: mengambil data langsung dari NHTSA Crash API bila akses HTTP diizinkan."""
    import urllib.request
    url = API_ENDPOINT.format(year=year)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
        "Accept": "application/json",
        "Referer": "https://crashviewer.nhtsa.dot.gov/CrashAPI",
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8", errors="replace"))


def extract_raw_sources(csv_path, api_path):
    """Tahap 1 ELT: mengekstrak seluruh sumber data dalam bentuk mentah (tanpa transformasi)."""
    log("EXTRACT  | Memulai ekstraksi data mentah dari sumber (tanpa cleaning/transformasi) ...")
    t0 = time.time()

    # 1. Sumber Kaggle CSV dibaca apa adanya sebagai string mentah (tanpa konversi tipe)
    df_main_raw = pd.read_csv(csv_path, dtype=str, keep_default_na=False, low_memory=False)
    main_records = df_main_raw.to_dict("records")

    # 2. Sumber NHTSA Crash API JSON dibaca apa adanya
    api_payload = json.loads(api_path.read_text(encoding="utf-8"))
    api_records = [
        item
        for group in api_payload.get("Results", [])
        for item in (group if isinstance(group, list) else [group])
    ]
    df_api_raw = pd.DataFrame(api_records)

    # 3. Referensi NHTSA GetStates dalam struktur mentah
    state_records = [
        {"State": code, "NHTSA_State_ID": vals[0], "NHTSA_State_Name": vals[1], "NHTSA_Region": vals[2]}
        for code, vals in NHTSA_STATES.items()
    ]
    df_states_raw = pd.DataFrame(state_records)

    # Pemeriksaan awal karakteristik (inspeksi saja, tidak mengubah data)
    missing_main = int((df_main_raw == "").sum().sum())
    dup_main_rows = int(df_main_raw.duplicated().sum())
    dup_main_id = int(df_main_raw["ID"].duplicated().sum())
    weather_raw_unique = int(df_main_raw["Weather_Condition"].nunique())
    na_weather_count = int(df_main_raw["Weather_Condition"].str.lower().str.contains("n/a", na=False).sum())

    missing_api = int(df_api_raw.isna().sum().sum())
    dup_api_exact = int(df_api_raw.duplicated().sum())
    dup_api_state_case = int(df_api_raw.duplicated(subset=["State", "St_Case"]).sum())

    elapsed = round(time.time() - t0, 2)
    info = {
        "main": {
            "source": str(csv_path.relative_to(PROJECT_ROOT)),
            "format": "CSV (UTF-8)",
            "rows": int(df_main_raw.shape[0]),
            "cols": int(df_main_raw.shape[1]),
            "columns": list(df_main_raw.columns),
            "raw_dtypes": {c: str(t) for c, t in df_main_raw.dtypes.items()},
            "missing_empty": missing_main,
            "duplicate_rows": dup_main_rows,
            "duplicate_id": dup_main_id,
            "weather_unique": weather_raw_unique,
            "weather_na_label_rows": na_weather_count,
            "sample": df_main_raw.head(5).to_dict("records"),
        },
        "api": {
            "source": str(api_path.relative_to(PROJECT_ROOT)),
            "endpoint": API_ENDPOINT.format(year="2020..2026"),
            "format": "JSON (NHTSA CrashAPI Results)",
            "rows": int(len(df_api_raw)),
            "cols": int(df_api_raw.shape[1]),
            "count_reported": int(api_payload.get("Count", len(df_api_raw))),
            "columns": list(df_api_raw.columns),
            "raw_dtypes": {c: str(t) for c, t in df_api_raw.dtypes.items()},
            "missing": missing_api,
            "duplicate_exact": dup_api_exact,
            "duplicate_state_case": dup_api_state_case,
            "sample": df_api_raw.head(5).to_dict("records"),
        },
        "states_ref": {
            "source": "NHTSA CrashAPI definitions/GetStates (cached)",
            "format": "JSON dictionary",
            "rows": int(len(df_states_raw)),
            "cols": int(df_states_raw.shape[1]),
        },
        "elapsed_seconds": elapsed,
    }

    log(f"EXTRACT  | Sumber 1 (Kaggle CSV): {info['main']['rows']:,} baris x {info['main']['cols']} kolom (raw string)")
    log(f"EXTRACT  | Sumber 2 (NHTSA Crash API JSON): {info['api']['rows']:,} baris x {info['api']['cols']} atribut")
    log(f"EXTRACT  | Sumber 3 (NHTSA GetStates): {info['states_ref']['rows']} entitas referensi state")
    log("EXTRACT  | Pemeriksaan awal (tanpa modifikasi):")
    log(f"EXTRACT  |   Kaggle CSV -> missing/kosong: {missing_main}, duplikat ID: {dup_main_id}, "
        f"variasi Weather_Condition: {weather_raw_unique} (termasuk {na_weather_count} 'N/A Precipitation')")
    log(f"EXTRACT  |   NHTSA API  -> missing: {missing_api}, duplikat eksak: {dup_api_exact}, "
        f"St_Case berulang lintas tahun: {dup_api_state_case:,}, format tanggal /Date(ms+0000)/")
    log(f"EXTRACT  | Selesai dalam {elapsed} detik")

    return main_records, api_records, state_records, df_main_raw, df_api_raw, info


raw_main_records, raw_api_records, raw_state_records, df_main_extract, df_api_extract, extract_info = extract_raw_sources(
    SOURCE_CSV, API_SNAPSHOT
)

print("\nStruktur Atribut Sumber Utama (Raw Extract):")
display(pd.DataFrame({
    "Kolom": extract_info["main"]["columns"],
    "Tipe_Raw": [extract_info["main"]["raw_dtypes"][c] for c in extract_info["main"]["columns"]],
    "Kosong": [int((df_main_extract[c] == "").sum()) for c in extract_info["main"]["columns"]],
}).head(10))
print("Contoh 5 baris raw Dataset Utama:")
display(df_main_extract.head(5))
print("Contoh 5 baris raw NHTSA Crash API JSON:")
display(df_api_extract.head(5))


[14:51:17] [INFO] EXTRACT  | Memulai ekstraksi data mentah dari sumber (tanpa cleaning/transformasi) ...


[14:51:24] [INFO] EXTRACT  | Sumber 1 (Kaggle CSV): 266,656 baris x 21 kolom (raw string)


[14:51:24] [INFO] EXTRACT  | Sumber 2 (NHTSA Crash API JSON): 8,798 baris x 9 atribut


[14:51:24] [INFO] EXTRACT  | Sumber 3 (NHTSA GetStates): 53 entitas referensi state


[14:51:24] [INFO] EXTRACT  | Pemeriksaan awal (tanpa modifikasi):


[14:51:24] [INFO] EXTRACT  |   Kaggle CSV -> missing/kosong: 0, duplikat ID: 0, variasi Weather_Condition: 81 (termasuk 187 'N/A Precipitation')


... (4 outputs left out)

Contoh 5 baris raw Dataset Utama:


,ID,Severity,Severity_Label,Start_Time,End_Time,Duration_Minutes,Hour,Day_of_Week,City,County,...,NHTSA_State_ID,NHTSA_State_Name,NHTSA_Region,NHTSA_Fatality_Risk_Tier,NHTSA_API_Latest_FARS_Year,Temperature(F),Humidity(%),Visibility(mi),Weather_Condition,Data_Source
0,NHTSA-2024-483866,4,Severe,2024-12-31 23:25:00,2025-01-01 02:25:00,180.0,23,Tuesday,Harris,Harris,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
1,NHTSA-2024-483853,4,Severe,2024-12-31 23:01:00,2025-01-01 02:01:00,180.0,23,Tuesday,Travis,Travis,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
2,NHTSA-2024-483739,4,Severe,2024-12-31 21:51:00,2025-01-01 00:51:00,180.0,21,Tuesday,Tarrant,Tarrant,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
3,NHTSA-2024-483871,4,Severe,2024-12-31 15:30:00,2024-12-31 18:30:00,180.0,15,Tuesday,Harris,Harris,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
4,NHTSA-2024-483790,4,Severe,2024-12-31 03:00:00,2024-12-31 06:00:00,180.0,3,Tuesday,Cameron,Cameron,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI


Contoh 5 baris raw NHTSA Crash API JSON:


,CountyName,CrashDate,Fatals,Peds,Persons,St_Case,State,StateName,TotalVehicles
0,ELMORE (51),/Date(1577847480000+0000)/,3,0,4,10001,1,Alabama,1
1,JEFFERSON (73),/Date(1577985480000+0000)/,1,0,6,10002,1,Alabama,4
2,SHELBY (117),/Date(1577976900000+0000)/,1,0,2,10003,1,Alabama,2
3,CALHOUN (15),/Date(1578064800000+0000)/,1,0,5,10004,1,Alabama,1
4,COOSA (37),/Date(1578098700000+0000)/,1,0,1,10005,1,Alabama,1


## 2. Load — Raw Data Loading ke Target Storage

Sesuai rancangan ELT Minggu 9 (*Airbyte → Supabase skema `raw` dalam kolom bertipe JSONB*):
- **Target storage**: Supabase (PostgreSQL), direpresentasikan secara lokal pada **SQLite 3.53.1** (`Dataset/supabase_local.db`) dengan pemisahan skema melalui penamaan tabel **`raw_*`** vs **`analytics_*`**.
- **Struktur tabel raw** mengikuti pola Airbyte Destinations v2:
  - `_airbyte_raw_id` (Primary Key berurut)
  - `_airbyte_extracted_at` (timestamp ekstraksi UTC)
  - `_airbyte_source` (identitas sumber data)
  - `_airbyte_data` (payload **JSON mentah utuh** tanpa pembersihan, tanpa pemfilteran baris invalid, dan tanpa pengubahan tipe data)
- Ketiga tabel raw yang dimuat secara terpisah dari hasil analitik:
  1. `raw_accidents` (setara `raw.accidents_raw` — **266.656 baris**)
  2. `raw_nhtsa_api_cases` (setara `raw.nhtsa_api_cases_raw` — **8.798 baris**, mencakup data kotor seperti tanggal sentinel `/Date(-62135596800000+0000)/` dan nama county `ELMORE (51)`)
  3. `raw_nhtsa_states` (setara `raw.nhtsa_states_raw` — **53 entitas**)


In [3]:
def load_raw_to_storage(main_records, api_records, state_records, db_path):
    """Tahap 2 ELT: memuat raw data dalam format JSON utuh ke tabel raw_* pada target storage."""
    log("LOAD     | Menghubungkan ke target storage (Supabase PostgreSQL / skema raw) ...")
    t0 = time.time()
    extracted_at = datetime.now(timezone.utc).isoformat()

    conn = open_sqlite_elt_conn(db_path)
    log(f"LOAD     | Koneksi berhasil -> SQLite {sqlite3.sqlite_version} | database: {db_path.name} | skema: raw")

    raw_specs = [
        ("raw_accidents", main_records, "kaggle_csv:US_Accidents_Cleaned_NHTSA_2020-2024.csv"),
        ("raw_nhtsa_api_cases", api_records, "nhtsa_api:GetCaseList"),
        ("raw_nhtsa_states", state_records, "nhtsa_api:GetStates"),
    ]

    counts = {}
    samples = {}
    for table_name, records, source_tag in raw_specs:
        conn.execute(f"DROP TABLE IF EXISTS {table_name}")
        conn.execute(f"""
            CREATE TABLE {table_name} (
                _airbyte_raw_id INTEGER PRIMARY KEY,
                _airbyte_extracted_at TEXT NOT NULL,
                _airbyte_source TEXT NOT NULL,
                _airbyte_data TEXT NOT NULL
            )
        """)
        rows_payload = [
            (idx, extracted_at, source_tag, json.dumps(rec, ensure_ascii=False))
            for idx, rec in enumerate(records, 1)
        ]
        conn.executemany(f"INSERT INTO {table_name} VALUES (?, ?, ?, ?)", rows_payload)
        conn.commit()

        stored_count = int(conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0])
        assert stored_count == len(records), f"Mismatch row count pada {table_name}: {stored_count} != {len(records)}"
        counts[table_name] = stored_count

        sample_df = pd.read_sql(
            f"SELECT _airbyte_raw_id, _airbyte_extracted_at, _airbyte_source, "
            f"SUBSTR(_airbyte_data, 1, 140) || '...' AS _airbyte_data_preview FROM {table_name} LIMIT 3",
            conn,
        )
        samples[table_name] = sample_df
        log(f"LOAD     | {stored_count:,} baris raw JSON berhasil dimuat ke raw.{table_name.replace('raw_', '')} ({table_name})")

    conn.close()
    elapsed = round(time.time() - t0, 2)
    log(f"LOAD     | Seluruh raw data berhasil disimpan & diverifikasi pembacaannya dalam {elapsed} detik")
    return {
        "database": db_path.name,
        "mode": "SQLite (representasi lokal Supabase PostgreSQL skema raw)",
        "counts": counts,
        "samples": samples,
        "elapsed_seconds": elapsed,
    }


def load_raw_to_supabase_pg(main_records, api_records, state_records, dsn):
    """Opsional: muat raw JSONB langsung ke Supabase PostgreSQL bila SUPABASE_DB_URL tersedia."""
    from sqlalchemy import create_engine
    from psycopg2.extras import execute_values

    engine = create_engine(dsn)
    extracted_at = datetime.now(timezone.utc).isoformat()
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
            cur.execute("CREATE SCHEMA IF NOT EXISTS analytics;")
        raw_conn.commit()

        for tbl, recs, src in [
            ("raw.accidents_raw", main_records, "kaggle_csv"),
            ("raw.nhtsa_api_cases_raw", api_records, "nhtsa_crash_api"),
            ("raw.nhtsa_states_raw", state_records, "nhtsa_get_states"),
        ]:
            with raw_conn.cursor() as cur:
                cur.execute(f"DROP TABLE IF EXISTS {tbl};")
                cur.execute(f"""
                    CREATE TABLE {tbl} (
                        _airbyte_raw_id BIGSERIAL PRIMARY KEY,
                        _airbyte_extracted_at TIMESTAMPTZ NOT NULL,
                        _airbyte_source TEXT NOT NULL,
                        _airbyte_data JSONB NOT NULL
                    );
                """)
                data = [(extracted_at, src, json.dumps(r)) for r in recs]
                for i in range(0, len(data), 5000):
                    execute_values(
                        cur,
                        f"INSERT INTO {tbl} (_airbyte_extracted_at, _airbyte_source, _airbyte_data) VALUES %s",
                        data[i : i + 5000],
                    )
            raw_conn.commit()
    finally:
        raw_conn.close()


# Ekspor raw_dataset.csv & raw_nhtsa_api_cases.csv sebagai artefak dataset mentah
df_main_extract.to_csv(RAW_MAIN_CSV, index=False)
df_api_extract.to_csv(RAW_API_CSV, index=False)
load_raw_info = load_raw_to_storage(raw_main_records, raw_api_records, raw_state_records, WAREHOUSE_DB)

print("\nVerifikasi pembacaan kembali raw_accidents dari storage:")
display(load_raw_info["samples"]["raw_accidents"])
print("Verifikasi pembacaan kembali raw_nhtsa_api_cases dari storage:")
display(load_raw_info["samples"]["raw_nhtsa_api_cases"])


[14:51:25] [INFO] LOAD     | Menghubungkan ke target storage (Supabase PostgreSQL / skema raw) ...


[14:51:25] [INFO] LOAD     | Koneksi berhasil -> SQLite 3.53.1 | database: supabase_local.db | skema: raw


[14:51:41] [INFO] LOAD     | 266,656 baris raw JSON berhasil dimuat ke raw.accidents (raw_accidents)


[14:51:41] [INFO] LOAD     | 8,798 baris raw JSON berhasil dimuat ke raw.nhtsa_api_cases (raw_nhtsa_api_cases)


[14:51:41] [INFO] LOAD     | 53 baris raw JSON berhasil dimuat ke raw.nhtsa_states (raw_nhtsa_states)


[14:51:41] [INFO] LOAD     | Seluruh raw data berhasil disimpan & diverifikasi pembacaannya dalam 15.97 detik



Verifikasi pembacaan kembali raw_accidents dari storage:


,_airbyte_raw_id,_airbyte_extracted_at,_airbyte_source,_airbyte_data_preview
0,1,2026-11-26T07:51:25.809547+00:00,kaggle_csv:US_Accidents_Cleaned_NHTSA_2020-202...,"{""ID"": ""NHTSA-2024-483866"", ""Severity"": ""4"", ""..."
1,2,2026-11-26T07:51:25.809547+00:00,kaggle_csv:US_Accidents_Cleaned_NHTSA_2020-202...,"{""ID"": ""NHTSA-2024-483853"", ""Severity"": ""4"", ""..."
2,3,2026-11-26T07:51:25.809547+00:00,kaggle_csv:US_Accidents_Cleaned_NHTSA_2020-202...,"{""ID"": ""NHTSA-2024-483739"", ""Severity"": ""4"", ""..."


Verifikasi pembacaan kembali raw_nhtsa_api_cases dari storage:


,_airbyte_raw_id,_airbyte_extracted_at,_airbyte_source,_airbyte_data_preview
0,1,2026-11-26T07:51:25.809547+00:00,nhtsa_api:GetCaseList,"{""CountyName"": ""ELMORE (51)"", ""CrashDate"": ""/D..."
1,2,2026-11-26T07:51:25.809547+00:00,nhtsa_api:GetCaseList,"{""CountyName"": ""JEFFERSON (73)"", ""CrashDate"": ..."
2,3,2026-11-26T07:51:25.809547+00:00,nhtsa_api:GetCaseList,"{""CountyName"": ""SHELBY (117)"", ""CrashDate"": ""/..."


## 3. Transform — Data Inspection, Cleaning & Transformation (In-Database SQL)

Pada pendekatan **ELT**, proses transformasi **tidak dilakukan di Python/pandas sebelum load**, melainkan
dieksekusi **langsung di dalam target storage** terhadap tabel `raw_*` yang sudah tersimpan (setara dengan
model SQL **dbt** yang mengirim perintah SQL ke mesin PostgreSQL Supabase).

### 3a. Permasalahan Data yang Teridentifikasi pada `raw_*` di Storage

| Sumber Raw | Masalah yang Ditemukan | Bukti pada Raw Storage | Penanganan SQL |
|---|---|---|---|
| `raw_accidents` | Data tersimpan sebagai payload JSON semi-terstruktur (`_airbyte_data`) | 1 kolom JSON per baris | Ekstraksi atribut via `json_extract()` (`->>` di Postgres) |
| `raw_accidents` | `Start_Time` & `End_Time` masih berupa string teks mentah | Tipe teks di dalam JSON | Konversi ke `datetime(...)` + filter non-null |
| `raw_accidents` | `Weather_Condition` memiliki **81** variasi label (termasuk **187** `"N/A Precipitation"`) | Kardinalitas tinggi untuk BI | Standardisasi ke **7 grup cuaca** + encoding `Weather_Code` (`0..6`) |
| `raw_accidents` | Kebutuhan atribut analitik turunan & validasi referensi/rentang | Belum ada kolom tanggal/weekend/durasi | `Accident_Date`, `Month`, `Is_Weekend`, `Duration_Category`, `JOIN raw_nhtsa_states`, dedup `ROW_NUMBER()` |
| `raw_nhtsa_api_cases` | Format tanggal Microsoft JSON epoch `"/Date(1577847480000+0000)/"` | 8.798 string `/Date(...)/` | Ekstrak epoch ms & konversi `datetime(ms/1000.0, 'unixepoch')` |
| `raw_nhtsa_api_cases` | **11** tanggal sentinel/invalid di luar 2020–2026 (`"/Date(-62135596800000+0000)/"` = tahun `0001`) | Terdeteksi via query SQL `strftime('%Y')` | Filter `YEAR BETWEEN 2020 AND 2026` (**11 baris dibuang**) |
| `raw_nhtsa_api_cases` | **8** baris memiliki nilai tidak logis (`Persons = 0` saat `Fatals = 1, Peds = 1`) | Terdeteksi via query SQL | Filter `Fatals >= 1 AND Persons >= 1 AND TotalVehicles >= 1 AND Peds >= 0` (**8 baris dibuang**) |
| `raw_nhtsa_api_cases` | **6.909** pasangan `(State, St_Case)` berulang antar tahun | `St_Case` hanya unik per tahun | Deduplikasi kunci komposit `(State_ID, St_Case, Crash_Date)` dengan `ROW_NUMBER()` |
| `raw_nhtsa_api_cases` | `CountyName` kotor berisi akhiran FIPS & huruf kapital (`"ELMORE (51)"`, `"RICHMOND (CITY) (760)"`) | Format tidak konsisten | `INITCAP(TRIM(RTRIM(RTRIM(CountyName, '0123456789)'), '(')))` |
| `raw_nhtsa_api_cases` | `State` masih berupa ID numerik FIPS (`1`, `51`) & belum ada `Fatality_Risk_Tier` | Belum selaras skema analitik | `INNER JOIN raw_nhtsa_states` + `LEFT JOIN` risk tier dari `analytics_accidents` |


In [4]:
SQL_ANALYTICS_ACCIDENTS = """
CREATE TABLE analytics_accidents AS
WITH states_dim AS (
    SELECT
        UPPER(TRIM(json_extract(_airbyte_data, '$.State'))) AS State,
        CAST(json_extract(_airbyte_data, '$.NHTSA_State_ID') AS INTEGER) AS NHTSA_State_ID_Ref,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_State_Name')) AS NHTSA_State_Name_Ref,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_Region')) AS NHTSA_Region_Ref
    FROM raw_nhtsa_states
),
parsed AS (
    SELECT
        _airbyte_raw_id,
        TRIM(json_extract(_airbyte_data, '$.ID')) AS ID,
        CAST(json_extract(_airbyte_data, '$.Severity') AS INTEGER) AS Severity,
        TRIM(json_extract(_airbyte_data, '$.Severity_Label')) AS Severity_Label,
        datetime(TRIM(json_extract(_airbyte_data, '$.Start_Time'))) AS Start_Time,
        datetime(TRIM(json_extract(_airbyte_data, '$.End_Time'))) AS End_Time,
        CAST(json_extract(_airbyte_data, '$.Duration_Minutes') AS REAL) AS Duration_Minutes,
        CAST(json_extract(_airbyte_data, '$.Hour') AS INTEGER) AS Hour,
        TRIM(json_extract(_airbyte_data, '$.Day_of_Week')) AS Day_of_Week,
        INITCAP(TRIM(json_extract(_airbyte_data, '$.City'))) AS City,
        INITCAP(TRIM(json_extract(_airbyte_data, '$.County'))) AS County,
        UPPER(TRIM(json_extract(_airbyte_data, '$.State'))) AS State,
        CAST(json_extract(_airbyte_data, '$.NHTSA_State_ID') AS INTEGER) AS Raw_NHTSA_State_ID,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_State_Name')) AS Raw_NHTSA_State_Name,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_Region')) AS Raw_NHTSA_Region,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_Fatality_Risk_Tier')) AS NHTSA_Fatality_Risk_Tier,
        CAST(json_extract(_airbyte_data, '$.NHTSA_API_Latest_FARS_Year') AS INTEGER) AS NHTSA_API_Latest_FARS_Year,
        CAST(json_extract(_airbyte_data, '$."Temperature(F)"') AS REAL) AS "Temperature(F)",
        CAST(json_extract(_airbyte_data, '$."Humidity(%)"') AS REAL) AS "Humidity(%)",
        CAST(json_extract(_airbyte_data, '$."Visibility(mi)"') AS REAL) AS "Visibility(mi)",
        TRIM(json_extract(_airbyte_data, '$.Weather_Condition')) AS Weather_Condition,
        TRIM(json_extract(_airbyte_data, '$.Data_Source')) AS Data_Source
    FROM raw_accidents
),
dedup AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY ID ORDER BY _airbyte_raw_id) AS _rn
    FROM parsed
    WHERE Start_Time IS NOT NULL AND End_Time IS NOT NULL
),
validated AS (
    SELECT
        d.ID, d.Severity, d.Severity_Label, d.Start_Time, d.End_Time,
        date(d.Start_Time) AS Accident_Date,
        CAST(strftime('%m', d.Start_Time) AS INTEGER) AS Month,
        d.Hour, d.Day_of_Week,
        CASE WHEN CAST(strftime('%w', d.Start_Time) AS INTEGER) IN (0, 6) THEN 1 ELSE 0 END AS Is_Weekend,
        d.Duration_Minutes,
        CASE
            WHEN d.Duration_Minutes < 30 THEN 'Short (<30m)'
            WHEN d.Duration_Minutes <= 120 THEN 'Medium (30-120m)'
            ELSE 'Long (>120m)'
        END AS Duration_Category,
        d.City, d.County, d.State,
        s.NHTSA_State_ID_Ref AS NHTSA_State_ID,
        s.NHTSA_State_Name_Ref AS NHTSA_State_Name,
        s.NHTSA_Region_Ref AS NHTSA_Region,
        d.NHTSA_Fatality_Risk_Tier,
        d.NHTSA_API_Latest_FARS_Year,
        d."Temperature(F)", d."Humidity(%)", d."Visibility(mi)",
        d.Weather_Condition,
        CASE
            WHEN lower(d.Weather_Condition) LIKE 'n/a%' OR lower(d.Weather_Condition) IN ('nan','none','unknown') THEN 'Other'
            WHEN lower(d.Weather_Condition) LIKE '%thunder%' OR lower(d.Weather_Condition) LIKE '%t-storm%'
              OR lower(d.Weather_Condition) LIKE '%storm%' OR lower(d.Weather_Condition) LIKE '%squall%'
              OR lower(d.Weather_Condition) LIKE '%hail%' THEN 'Storm'
            WHEN lower(d.Weather_Condition) LIKE '%snow%' OR lower(d.Weather_Condition) LIKE '%sleet%'
              OR lower(d.Weather_Condition) LIKE '%freezing%' OR lower(d.Weather_Condition) LIKE '%wintry%'
              OR lower(d.Weather_Condition) LIKE '%ice%' THEN 'Snow/Ice'
            WHEN lower(d.Weather_Condition) LIKE '%rain%' OR lower(d.Weather_Condition) LIKE '%drizzle%'
              OR lower(d.Weather_Condition) LIKE '%shower%' OR lower(d.Weather_Condition) LIKE '%precipitation%' THEN 'Rain'
            WHEN lower(d.Weather_Condition) LIKE '%fog%' OR lower(d.Weather_Condition) LIKE '%mist%'
              OR lower(d.Weather_Condition) LIKE '%haze%' OR lower(d.Weather_Condition) LIKE '%smoke%'
              OR lower(d.Weather_Condition) LIKE '%dust%' THEN 'Fog/Haze'
            WHEN lower(d.Weather_Condition) LIKE '%cloudy%' OR lower(d.Weather_Condition) LIKE '%overcast%' THEN 'Cloudy'
            WHEN lower(d.Weather_Condition) LIKE '%fair%' OR lower(d.Weather_Condition) LIKE '%clear%' THEN 'Clear'
            ELSE 'Other'
        END AS Weather_Group,
        d.Data_Source
    FROM dedup d
    INNER JOIN states_dim s ON d.State = s.State
    WHERE d._rn = 1
      AND d.Severity BETWEEN 1 AND 4
      AND d.Duration_Minutes BETWEEN 0 AND 1440
      AND d."Temperature(F)" BETWEEN -40 AND 125
      AND d."Humidity(%)" BETWEEN 0 AND 100
      AND d."Visibility(mi)" BETWEEN 0 AND 50
)
SELECT
    ID, Severity, Severity_Label, Start_Time, End_Time,
    Accident_Date, Month, Hour, Day_of_Week, Is_Weekend,
    Duration_Minutes, Duration_Category, City, County, State,
    NHTSA_State_ID, NHTSA_State_Name, NHTSA_Region,
    NHTSA_Fatality_Risk_Tier, NHTSA_API_Latest_FARS_Year,
    "Temperature(F)", "Humidity(%)", "Visibility(mi)",
    Weather_Condition, Weather_Group,
    CASE Weather_Group
        WHEN 'Clear' THEN 0 WHEN 'Cloudy' THEN 1 WHEN 'Fog/Haze' THEN 2
        WHEN 'Other' THEN 3 WHEN 'Rain' THEN 4 WHEN 'Snow/Ice' THEN 5
        WHEN 'Storm' THEN 6
    END AS Weather_Code,
    Data_Source
FROM validated
ORDER BY Start_Time DESC, ID ASC
"""

SQL_ANALYTICS_API_CASES = """
CREATE TABLE analytics_nhtsa_api_cases AS
WITH states_dim AS (
    SELECT
        CAST(json_extract(_airbyte_data, '$.NHTSA_State_ID') AS INTEGER) AS State_ID,
        UPPER(TRIM(json_extract(_airbyte_data, '$.State'))) AS State,
        TRIM(json_extract(_airbyte_data, '$.NHTSA_Region')) AS NHTSA_Region
    FROM raw_nhtsa_states
),
risk_dim AS (
    SELECT DISTINCT State, NHTSA_Fatality_Risk_Tier FROM analytics_accidents
),
parsed AS (
    SELECT
        _airbyte_raw_id,
        CAST(json_extract(_airbyte_data, '$.St_Case') AS INTEGER) AS St_Case,
        CAST(json_extract(_airbyte_data, '$.State') AS INTEGER) AS State_ID,
        INITCAP(TRIM(json_extract(_airbyte_data, '$.StateName'))) AS StateName,
        INITCAP(TRIM(RTRIM(RTRIM(TRIM(json_extract(_airbyte_data, '$.CountyName')), '0123456789)'), '('))) AS CountyName,
        datetime(
            CAST(
                SUBSTR(
                    json_extract(_airbyte_data, '$.CrashDate'),
                    7,
                    CASE
                        WHEN INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 7), '+') > 0
                            THEN INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 7), '+') - 1
                        ELSE INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 8), '-')
                    END
                ) AS INTEGER
            ) / 1000.0,
            'unixepoch'
        ) AS Crash_Date,
        CAST(json_extract(_airbyte_data, '$.Fatals') AS INTEGER) AS Fatals,
        CAST(json_extract(_airbyte_data, '$.Peds') AS INTEGER) AS Peds,
        CAST(json_extract(_airbyte_data, '$.Persons') AS INTEGER) AS Persons,
        CAST(json_extract(_airbyte_data, '$.TotalVehicles') AS INTEGER) AS TotalVehicles
    FROM raw_nhtsa_api_cases
),
valid_dates AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY State_ID, St_Case, Crash_Date ORDER BY _airbyte_raw_id) AS _rn
    FROM parsed
    WHERE Crash_Date IS NOT NULL AND CAST(strftime('%Y', Crash_Date) AS INTEGER) BETWEEN 2020 AND 2026
)
SELECT
    v.St_Case,
    s.State,
    v.StateName,
    v.CountyName,
    v.Crash_Date,
    v.Fatals,
    v.Peds,
    v.Persons,
    v.TotalVehicles,
    s.NHTSA_Region,
    COALESCE(r.NHTSA_Fatality_Risk_Tier, 'Low') AS Fatality_Risk_Tier,
    'NHTSA_CrashAPI_Live' AS Data_Source
FROM valid_dates v
INNER JOIN states_dim s ON v.State_ID = s.State_ID
LEFT JOIN risk_dim r ON s.State = r.State
WHERE v._rn = 1
  AND v.Fatals >= 1 AND v.Persons >= 1 AND v.TotalVehicles >= 1 AND v.Peds >= 0
"""


def inspect_raw_in_storage(conn):
    """Tahap 3a ELT: menginspeksi kualitas raw_* langsung di dalam target storage via SQL."""
    log("TRANSFORM| (3a) Data Inspection pada tabel raw_* di dalam storage ...")
    q_main = conn.execute("""
        SELECT
            COUNT(*) AS total_raw,
            SUM(CASE WHEN json_extract(_airbyte_data, '$.ID') IS NULL OR TRIM(json_extract(_airbyte_data, '$.ID')) = '' THEN 1 ELSE 0 END) AS null_id,
            SUM(CASE WHEN datetime(json_extract(_airbyte_data, '$.Start_Time')) IS NULL THEN 1 ELSE 0 END) AS bad_datetime,
            COUNT(DISTINCT json_extract(_airbyte_data, '$.Weather_Condition')) AS weather_variants,
            SUM(CASE WHEN lower(json_extract(_airbyte_data, '$.Weather_Condition')) LIKE 'n/a%' THEN 1 ELSE 0 END) AS na_weather_rows,
            SUM(CASE WHEN CAST(json_extract(_airbyte_data, '$.Severity') AS INTEGER) NOT BETWEEN 1 AND 4 THEN 1 ELSE 0 END) AS invalid_severity
        FROM raw_accidents
    """).fetchone()

    q_api = conn.execute("""
        WITH parsed AS (
            SELECT
                CAST(json_extract(_airbyte_data, '$.State') AS INTEGER) AS state_id,
                CAST(json_extract(_airbyte_data, '$.St_Case') AS INTEGER) AS st_case,
                json_extract(_airbyte_data, '$.CountyName') AS county_raw,
                datetime(
                    CAST(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 7,
                        CASE WHEN INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 7), '+') > 0
                             THEN INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 7), '+') - 1
                             ELSE INSTR(SUBSTR(json_extract(_airbyte_data, '$.CrashDate'), 8), '-') END) AS INTEGER) / 1000.0,
                    'unixepoch'
                ) AS crash_dt,
                CAST(json_extract(_airbyte_data, '$.Fatals') AS INTEGER) AS fatals,
                CAST(json_extract(_airbyte_data, '$.Persons') AS INTEGER) AS persons,
                CAST(json_extract(_airbyte_data, '$.TotalVehicles') AS INTEGER) AS vehicles,
                CAST(json_extract(_airbyte_data, '$.Peds') AS INTEGER) AS peds
            FROM raw_nhtsa_api_cases
        )
        SELECT
            COUNT(*) AS total_raw,
            SUM(CASE WHEN crash_dt IS NULL OR CAST(strftime('%Y', crash_dt) AS INTEGER) NOT BETWEEN 2020 AND 2026 THEN 1 ELSE 0 END) AS bad_date_rows,
            SUM(CASE WHEN (CAST(strftime('%Y', crash_dt) AS INTEGER) BETWEEN 2020 AND 2026)
                      AND NOT (fatals >= 1 AND persons >= 1 AND vehicles >= 1 AND peds >= 0) THEN 1 ELSE 0 END) AS invalid_metric_rows,
            SUM(CASE WHEN county_raw LIKE '%(%' THEN 1 ELSE 0 END) AS dirty_county_suffix_rows
        FROM parsed
    """).fetchone()

    dup_main_id = int(conn.execute("""
        SELECT COUNT(*) - COUNT(DISTINCT json_extract(_airbyte_data, '$.ID')) FROM raw_accidents
    """).fetchone()[0])

    dup_api_composite = int(conn.execute("""
        SELECT COUNT(*) - COUNT(DISTINCT
            json_extract(_airbyte_data, '$.State') || '|' ||
            json_extract(_airbyte_data, '$.St_Case') || '|' ||
            json_extract(_airbyte_data, '$.CrashDate')
        ) FROM raw_nhtsa_api_cases
    """).fetchone()[0])

    report = {
        "raw_accidents": {
            "total_raw": int(q_main[0]), "null_id": int(q_main[1]), "bad_datetime": int(q_main[2]),
            "weather_variants": int(q_main[3]), "na_weather_rows": int(q_main[4]),
            "invalid_severity": int(q_main[5]), "duplicate_id": dup_main_id,
        },
        "raw_nhtsa_api_cases": {
            "total_raw": int(q_api[0]), "bad_date_rows": int(q_api[1]),
            "invalid_metric_rows": int(q_api[2]), "dirty_county_suffix_rows": int(q_api[3]),
            "duplicate_composite": dup_api_composite,
        },
    }
    log(f"TRANSFORM|   raw_accidents: {report['raw_accidents']['total_raw']:,} baris | "
        f"weather={report['raw_accidents']['weather_variants']} variasi ({report['raw_accidents']['na_weather_rows']} N/A) | "
        f"duplikat ID={dup_main_id}")
    log(f"TRANSFORM|   raw_nhtsa_api_cases: {report['raw_nhtsa_api_cases']['total_raw']:,} baris | "
        f"tanggal sentinel invalid={report['raw_nhtsa_api_cases']['bad_date_rows']} | "
        f"nilai metrik invalid={report['raw_nhtsa_api_cases']['invalid_metric_rows']} | "
        f"CountyName berakhiran FIPS={report['raw_nhtsa_api_cases']['dirty_county_suffix_rows']:,}")
    return report


def transform_in_storage(db_path):
    """Tahap 3b-3e ELT: menjalankan SQL transformasi di dalam database -> tabel analytics_*."""
    log("TRANSFORM| Menjalankan SQL transformasi di dalam target storage (raw_* -> analytics_*) ...")
    t0 = time.time()
    conn = open_sqlite_elt_conn(db_path)

    inspection = inspect_raw_in_storage(conn)

    # 3b-3c. SQL Transform: analytics_accidents
    t_main = time.time()
    conn.execute("DROP TABLE IF EXISTS analytics_accidents")
    conn.execute(SQL_ANALYTICS_ACCIDENTS)
    conn.commit()
    main_elapsed = round(time.time() - t_main, 2)
    log(f"TRANSFORM| (3b-c) SQL analytics_accidents selesai dieksekusi di storage ({main_elapsed}s)")

    # 3b-3c. SQL Transform: analytics_nhtsa_api_cases
    t_api = time.time()
    conn.execute("DROP TABLE IF EXISTS analytics_nhtsa_api_cases")
    conn.execute(SQL_ANALYTICS_API_CASES)
    conn.commit()
    api_elapsed = round(time.time() - t_api, 2)
    log(f"TRANSFORM| (3b-c) SQL analytics_nhtsa_api_cases selesai dieksekusi di storage ({api_elapsed}s)")

    # 3d. Data Validation langsung di storage
    rows_main_out = int(conn.execute("SELECT COUNT(*) FROM analytics_accidents").fetchone()[0])
    cols_main_out = [r[1] for r in conn.execute("PRAGMA table_info(analytics_accidents)").fetchall()]
    nulls_main_out = int(conn.execute(
        "SELECT " + " + ".join(f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END)' for c in cols_main_out) + " FROM analytics_accidents"
    ).fetchone()[0])
    dup_main_out = int(conn.execute("SELECT COUNT(*) - COUNT(DISTINCT ID) FROM analytics_accidents").fetchone()[0])
    weather_after = int(conn.execute("SELECT COUNT(DISTINCT Weather_Group) FROM analytics_accidents").fetchone()[0])
    weather_counts = dict(conn.execute("SELECT Weather_Group, COUNT(*) FROM analytics_accidents GROUP BY Weather_Group ORDER BY COUNT(*) DESC").fetchall())
    checksum_main = int(conn.execute("SELECT SUM(Severity) FROM analytics_accidents").fetchone()[0])

    rows_api_out = int(conn.execute("SELECT COUNT(*) FROM analytics_nhtsa_api_cases").fetchone()[0])
    cols_api_out = [r[1] for r in conn.execute("PRAGMA table_info(analytics_nhtsa_api_cases)").fetchall()]
    nulls_api_out = int(conn.execute(
        "SELECT " + " + ".join(f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END)' for c in cols_api_out) + " FROM analytics_nhtsa_api_cases"
    ).fetchone()[0])
    dup_api_out = int(conn.execute(
        "SELECT COUNT(*) - COUNT(DISTINCT State || '|' || St_Case || '|' || Crash_Date) FROM analytics_nhtsa_api_cases"
    ).fetchone()[0])
    checksum_api = int(conn.execute("SELECT SUM(St_Case) FROM analytics_nhtsa_api_cases").fetchone()[0])

    # 3e. Pastikan tabel raw_* tetap utuh dan terpisah setelah Transform
    raw_main_after = int(conn.execute("SELECT COUNT(*) FROM raw_accidents").fetchone()[0])
    raw_api_after = int(conn.execute("SELECT COUNT(*) FROM raw_nhtsa_api_cases").fetchone()[0])
    raw_states_after = int(conn.execute("SELECT COUNT(*) FROM raw_nhtsa_states").fetchone()[0])

    assert nulls_main_out == 0 and dup_main_out == 0
    assert nulls_api_out == 0 and dup_api_out == 0
    assert weather_after == 7
    assert raw_main_after == inspection["raw_accidents"]["total_raw"]
    assert raw_api_after == inspection["raw_nhtsa_api_cases"]["total_raw"]

    total_elapsed = round(time.time() - t0, 2)
    log(f"TRANSFORM| (3d) Validasi analytics_accidents: {rows_main_out:,} baris x {len(cols_main_out)} kolom | "
        f"null={nulls_main_out} | duplikat={dup_main_out} | cuaca=81->{weather_after} grup | checksum={checksum_main:,}")
    log(f"TRANSFORM| (3d) Validasi analytics_nhtsa_api_cases: {rows_api_out:,} baris x {len(cols_api_out)} kolom | "
        f"null={nulls_api_out} | duplikat={dup_api_out} | checksum={checksum_api:,}")
    log(f"TRANSFORM| (3e) Bukti isolasi raw: raw_accidents={raw_main_after:,} baris & raw_nhtsa_api_cases={raw_api_after:,} baris tetap utuh!")

    df_acc_sample = pd.read_sql("SELECT * FROM analytics_accidents LIMIT 5", conn)
    df_api_sample = pd.read_sql("SELECT * FROM analytics_nhtsa_api_cases LIMIT 5", conn)
    conn.close()

    return {
        "inspection": inspection,
        "main": {
            "rows_in": inspection["raw_accidents"]["total_raw"],
            "cols_in": 21,
            "rows_out": rows_main_out,
            "cols_out": len(cols_main_out),
            "columns": cols_main_out,
            "nulls_after": nulls_main_out,
            "duplicates_after": dup_main_out,
            "weather_categories_before": inspection["raw_accidents"]["weather_variants"],
            "weather_categories_after": weather_after,
            "weather_group_counts": {str(k): int(v) for k, v in weather_counts.items()},
            "checksum_severity": checksum_main,
            "elapsed_seconds": main_elapsed,
        },
        "api": {
            "rows_in": inspection["raw_nhtsa_api_cases"]["total_raw"],
            "cols_in": 9,
            "rows_out": rows_api_out,
            "cols_out": len(cols_api_out),
            "columns": cols_api_out,
            "removed_bad_date": inspection["raw_nhtsa_api_cases"]["bad_date_rows"],
            "removed_invalid_metrics": inspection["raw_nhtsa_api_cases"]["invalid_metric_rows"],
            "removed_duplicate": inspection["raw_nhtsa_api_cases"]["duplicate_composite"],
            "nulls_after": nulls_api_out,
            "duplicates_after": dup_api_out,
            "checksum_st_case": checksum_api,
            "elapsed_seconds": api_elapsed,
        },
        "raw_preserved": {
            "raw_accidents": raw_main_after,
            "raw_nhtsa_api_cases": raw_api_after,
            "raw_nhtsa_states": raw_states_after,
        },
        "elapsed_seconds": total_elapsed,
        "samples": {"accidents": df_acc_sample, "api": df_api_sample},
    }


transform_info = transform_in_storage(WAREHOUSE_DB)
display(transform_info["samples"]["accidents"])
display(transform_info["samples"]["api"])


[14:51:41] [INFO] TRANSFORM| Menjalankan SQL transformasi di dalam target storage (raw_* -> analytics_*) ...


[14:51:41] [INFO] TRANSFORM| (3a) Data Inspection pada tabel raw_* di dalam storage ...


[14:51:55] [INFO] TRANSFORM|   raw_accidents: 266,656 baris | weather=81 variasi (187 N/A) | duplikat ID=0


[14:51:55] [INFO] TRANSFORM|   raw_nhtsa_api_cases: 8,798 baris | tanggal sentinel invalid=11 | nilai metrik invalid=8 | CountyName berakhiran FIPS=8,798


[14:52:03] [INFO] TRANSFORM| (3b-c) SQL analytics_accidents selesai dieksekusi di storage (8.44s)


[14:52:04] [INFO] TRANSFORM| (3b-c) SQL analytics_nhtsa_api_cases selesai dieksekusi di storage (0.18s)


[14:52:06] [INFO] TRANSFORM| (3d) Validasi analytics_accidents: 266,656 baris x 27 kolom | null=0 | duplikat=0 | cuaca=81->7 grup | checksum=576,217


[14:52:06] [INFO] TRANSFORM| (3d) Validasi analytics_nhtsa_api_cases: 8,779 baris x 12 kolom | null=0 | duplikat=0 | checksum=2,269,732,984


[14:52:06] [INFO] TRANSFORM| (3e) Bukti isolasi raw: raw_accidents=266,656 baris & raw_nhtsa_api_cases=8,798 baris tetap utuh!


,ID,Severity,Severity_Label,Start_Time,End_Time,Accident_Date,Month,Hour,Day_of_Week,Is_Weekend,...,NHTSA_Region,NHTSA_Fatality_Risk_Tier,NHTSA_API_Latest_FARS_Year,Temperature(F),Humidity(%),Visibility(mi),Weather_Condition,Weather_Group,Weather_Code,Data_Source
0,NHTSA-2024-483866,4,Severe,2024-12-31 23:25:00,2025-01-01 02:25:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
1,NHTSA-2024-483853,4,Severe,2024-12-31 23:01:00,2025-01-01 02:01:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
2,NHTSA-2024-483739,4,Severe,2024-12-31 21:51:00,2025-01-01 00:51:00,2024-12-31,12,21,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
3,NHTSA-2024-483871,4,Severe,2024-12-31 15:30:00,2024-12-31 18:30:00,2024-12-31,12,15,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
4,NHTSA-2024-483790,4,Severe,2024-12-31 03:00:00,2024-12-31 06:00:00,2024-12-31,12,3,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI


,St_Case,State,StateName,CountyName,Crash_Date,Fatals,Peds,Persons,TotalVehicles,NHTSA_Region,Fatality_Risk_Tier,Data_Source
0,10001,AL,Alabama,Butler,2023-01-04 22:20:00,1,0,1,1,South,Moderate,NHTSA_CrashAPI_Live
1,10001,AL,Alabama,Elmore,2020-01-01 02:58:00,3,0,4,1,South,Moderate,NHTSA_CrashAPI_Live
2,10001,AL,Alabama,Pickens,2022-01-01 12:30:00,1,0,3,2,South,Moderate,NHTSA_CrashAPI_Live
3,10001,AL,Alabama,St. Clair,2021-02-12 22:10:00,2,0,3,2,South,Moderate,NHTSA_CrashAPI_Live
4,10001,AL,Alabama,Tuscaloosa,2024-01-01 10:30:00,1,0,1,1,South,Moderate,NHTSA_CrashAPI_Live


In [5]:
comparison = pd.DataFrame([
    ["Dataset Utama — Jumlah baris", f"{transform_info['main']['rows_in']:,}", f"{transform_info['main']['rows_out']:,}", "0 (seluruh baris valid)"],
    ["Dataset Utama — Jumlah kolom", transform_info["main"]["cols_in"], transform_info["main"]["cols_out"], "+6 fitur analitik"],
    ["Dataset Utama — Format waktu", "str (dalam JSON raw)", "TIMESTAMP (SQLite/PG)", "Konversi SQL datetime()"],
    ["Dataset Utama — Kategori cuaca", transform_info["main"]["weather_categories_before"], transform_info["main"]["weather_categories_after"], "-74 (standardisasi 7 grup)"],
    ["Dataset API — Jumlah baris", f"{transform_info['api']['rows_in']:,}", f"{transform_info['api']['rows_out']:,}", "-19 (11 tanggal sentinel + 8 Persons=0)"],
    ["Dataset API — Jumlah kolom", transform_info["api"]["cols_in"], transform_info["api"]["cols_out"], "+3 (State, Region, Risk Tier)"],
    ["Dataset API — Format tanggal", "/Date(ms+0000)/ (JSON)", "TIMESTAMP UTC", "Parsing epoch ms SQL"],
    ["Penyimpanan Raw (tetap utuh)", "raw_accidents (266,656) & raw_nhtsa_api_cases (8,798)", "Tetap terpisah di raw_*", "0 baris tertimpa/hilang"],
], columns=["Aspek", "Kondisi di Raw Storage", "Kondisi di Analytics Storage", "Perubahan"])
display(comparison)
log("TRANSFORM| Tabel perbandingan raw vs analytics ditampilkan di atas")


,Aspek,Kondisi di Raw Storage,Kondisi di Analytics Storage,Perubahan
0,Dataset Utama — Jumlah baris,"266,656","266,656",0 (seluruh baris valid)
1,Dataset Utama — Jumlah kolom,21,27,+6 fitur analitik
2,Dataset Utama — Format waktu,str (dalam JSON raw),TIMESTAMP (SQLite/PG),Konversi SQL datetime()
3,Dataset Utama — Kategori cuaca,81,7,-74 (standardisasi 7 grup)
4,Dataset API — Jumlah baris,"8,798","8,779",-19 (11 tanggal sentinel + 8 Persons=0)
5,Dataset API — Jumlah kolom,9,12,"+3 (State, Region, Risk Tier)"
6,Dataset API — Format tanggal,/Date(ms+0000)/ (JSON),TIMESTAMP UTC,Parsing epoch ms SQL
7,Penyimpanan Raw (tetap utuh),"raw_accidents (266,656) & raw_nhtsa_api_cases ...",Tetap terpisah di raw_*,0 baris tertimpa/hilang


[14:52:06] [INFO] TRANSFORM| Tabel perbandingan raw vs analytics ditampilkan di atas


## 4. Integrasi ELT Pipeline

Fungsi `run_elt_pipeline()` mengintegrasikan seluruh tahapan secara sekuensial:
**Extract (Raw) → Load (Raw JSON ke `raw_*`) → Transform (SQL di dalam Storage ke `analytics_*`) → Export & Validasi**.

Tahap **Transform** baru dijalankan setelah fungsi **Load** memastikan seluruh baris mentah berhasil tersimpan dan terbaca kembali dari target storage. Apabila suatu tahap mengalami kegagalan, pipeline langsung dihentikan dan dicatat pada log (selaras dengan mekanisme DAG Apache Airflow).

```mermaid
flowchart LR
    S1[(Kaggle CSV<br/>266.656 baris)] -->|1. Extract raw| E[Extract Layer<br/>tanpa cleaning]
    S2[(NHTSA Crash API<br/>8.798 kasus JSON)] -->|1. Extract raw| E
    S3[(NHTSA GetStates<br/>53 entitas)] -->|1. Extract raw| E
    E -->|2. Load JSON apa adanya| RAW[(Target Storage: skema raw<br/>raw_accidents, raw_nhtsa_api_cases,<br/>raw_nhtsa_states)]
    RAW -->|3. In-DB SQL Transform<br/>json_extract, parse epoch,<br/>dedup, clean, join, encode| ANA[(Target Storage: skema analytics<br/>analytics_accidents 266.656,<br/>analytics_nhtsa_api_cases 8.779)]
    ANA --> BI[Power BI Dashboard]
    AF[Apache Airflow DAG] -.orkestrasi E -> L -> T.-> E
    AF -.-> RAW
    AF -.-> ANA
```


In [6]:
def export_analytics_csv(db_path, out_main, out_api):
    conn = open_sqlite_elt_conn(db_path)
    df_acc = pd.read_sql("SELECT * FROM analytics_accidents", conn)
    df_api = pd.read_sql("SELECT * FROM analytics_nhtsa_api_cases", conn)
    conn.close()
    df_acc.to_csv(out_main, index=False)
    df_api.to_csv(out_api, index=False)
    return len(df_acc), len(df_api)


def run_elt_pipeline():
    log("=== ELT PIPELINE DIMULAI (Extract -> Load Raw -> Transform In-DB) ===")
    started = time.time()
    try:
        # 1. EXTRACT
        main_recs, api_recs, state_recs, df_m_raw, df_a_raw, e_info = extract_raw_sources(
            SOURCE_CSV, API_SNAPSHOT
        )
        df_m_raw.to_csv(RAW_MAIN_CSV, index=False)
        df_a_raw.to_csv(RAW_API_CSV, index=False)

        # 2. LOAD RAW
        l_info = load_raw_to_storage(main_recs, api_recs, state_recs, WAREHOUSE_DB)

        # Gate invariant: Transform hanya berjalan setelah raw counts terverifikasi
        assert l_info["counts"]["raw_accidents"] == e_info["main"]["rows"]
        assert l_info["counts"]["raw_nhtsa_api_cases"] == e_info["api"]["rows"]
        assert l_info["counts"]["raw_nhtsa_states"] == e_info["states_ref"]["rows"]

        # 3. TRANSFORM IN-DATABASE
        t_info = transform_in_storage(WAREHOUSE_DB)

        # 4. EXPORT ANALYTICS SNAPSHOTS
        exp_main, exp_api = export_analytics_csv(WAREHOUSE_DB, TRANSFORMED_CSV, API_CASES_CSV)
        log(f"INTEGRASI| Ekspor dataset hasil transformasi: {TRANSFORMED_CSV.name} ({exp_main:,}), {API_CASES_CSV.name} ({exp_api:,})")
    except Exception as exc:
        log(f"ELT PIPELINE GAGAL: {type(exc).__name__}: {exc}", "ERROR")
        raise

    total_elapsed = round(time.time() - started, 2)
    metrics = {
        "group": "Kelompok 8",
        "architecture": "ELT (Extract -> Load Raw JSON -> Transform In-DB SQL)",
        "extract": {k: v for k, v in e_info.items() if k != "sample"},
        "load_raw": {k: v for k, v in l_info.items() if k != "samples"},
        "transform_sql": {k: v for k, v in t_info.items() if k != "samples"},
        "pipeline": {
            "elapsed_seconds": total_elapsed,
            "status": "BERHASIL",
            "raw_main_file": RAW_MAIN_CSV.name,
            "raw_api_file": RAW_API_CSV.name,
            "transformed_file": TRANSFORMED_CSV.name,
            "api_file": API_CASES_CSV.name,
            "storage_file": WAREHOUSE_DB.name,
        },
    }
    METRICS_JSON.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    log(f"=== ELT PIPELINE SELESAI DALAM {total_elapsed} DETIK ===")
    LOG_FILE.write_text("\n".join(LOGS) + "\n", encoding="utf-8")
    return metrics


metrics = run_elt_pipeline()


[14:52:06] [INFO] === ELT PIPELINE DIMULAI (Extract -> Load Raw -> Transform In-DB) ===


[14:52:06] [INFO] EXTRACT  | Memulai ekstraksi data mentah dari sumber (tanpa cleaning/transformasi) ...


[14:52:13] [INFO] EXTRACT  | Sumber 1 (Kaggle CSV): 266,656 baris x 21 kolom (raw string)


[14:52:13] [INFO] EXTRACT  | Sumber 2 (NHTSA Crash API JSON): 8,798 baris x 9 atribut


[14:52:13] [INFO] EXTRACT  | Sumber 3 (NHTSA GetStates): 53 entitas referensi state


[14:52:13] [INFO] EXTRACT  | Pemeriksaan awal (tanpa modifikasi):


... (16 outputs left out)

[14:52:47] [INFO] TRANSFORM| (3d) Validasi analytics_nhtsa_api_cases: 8,779 baris x 12 kolom | null=0 | duplikat=0 | checksum=2,269,732,984


[14:52:47] [INFO] TRANSFORM| (3e) Bukti isolasi raw: raw_accidents=266,656 baris & raw_nhtsa_api_cases=8,798 baris tetap utuh!


[14:52:53] [INFO] INTEGRASI| Ekspor dataset hasil transformasi: transformed_dataset.csv (266,656), nhtsa_api_cases.csv (8,779)


[14:52:53] [INFO] === ELT PIPELINE SELESAI DALAM 46.98 DETIK ===


## 5. ELT Pipeline Testing and Validation

Pengujian dilakukan terhadap empat aspek wajib: **Extraction Testing**, **Loading Testing**, **Transformation Testing**, dan **End-to-End Testing**, dilengkapi tabel perbandingan jumlah baris pada setiap tahapan serta bukti bahwa tabel raw tetap dipertahankan terpisah.


In [7]:
em = metrics["extract"]["main"]
ea = metrics["extract"]["api"]
es = metrics["extract"]["states_ref"]
lr = metrics["load_raw"]
tm = metrics["transform_sql"]["main"]
ta = metrics["transform_sql"]["api"]
rp = metrics["transform_sql"]["raw_preserved"]

# Uji koneksi & pembacaan kembali kedua layer (raw vs analytics) langsung dari SQLite
conn_test = open_sqlite_elt_conn(WAREHOUSE_DB)
tables_in_db = [r[0] for r in conn_test.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
raw_sample_readback = conn_test.execute("SELECT COUNT(*) FROM raw_nhtsa_api_cases WHERE _airbyte_data LIKE '%-62135596800000%'").fetchone()[0]
analytics_sample_readback = conn_test.execute("SELECT COUNT(*) FROM analytics_nhtsa_api_cases WHERE CAST(strftime('%Y', Crash_Date) AS INTEGER) < 2020").fetchone()[0]
conn_test.close()

test_rows = [
    ["Extract", "Pembacaan sumber utama (raw)", "266,656 baris x 21 kolom tanpa cleaning",
     f"{em['rows']:,} baris x {em['cols']} kolom (raw str)", "Berhasil"],
    ["Extract", "Pembacaan NHTSA Crash API (raw)", "8,798 kasus JSON + 53 state referensi",
     f"{ea['rows']:,} kasus JSON, {es['rows']} state", "Berhasil"],
    ["Load", "Koneksi & struktur skema raw", "Tabel raw_* terbentuk dengan kolom JSON",
     f"SQLite {WAREHOUSE_DB.name}: raw_accidents, raw_nhtsa_api_cases, raw_nhtsa_states", "Berhasil"],
    ["Load", "Penyimpanan raw data utuh", "Jumlah raw di storage = hasil Extract",
     f"main: {lr['counts']['raw_accidents']:,}, api: {lr['counts']['raw_nhtsa_api_cases']:,}, states: {lr['counts']['raw_nhtsa_states']}", "Berhasil"],
    ["Load", "Pembacaan kembali raw data", "Raw JSON (termasuk data kotor) terbaca ulang",
     f"11 tanggal sentinel masih utuh di raw_nhtsa_api_cases (={raw_sample_readback})", "Berhasil"],
    ["Transform", "SQL cleaning & transform (utama)", "JSON diekstrak, tipe tanggal, 7 grup cuaca, 27 kolom",
     f"{tm['rows_out']:,} baris x {tm['cols_out']} kolom | null={tm['nulls_after']} | cuaca={tm['weather_categories_after']} grup", "Berhasil"],
    ["Transform", "SQL cleaning & transform (API)", "Epoch UTC parsed, CountyName bersih, buang 19 baris invalid",
     f"{ta['rows_in']:,} -> {ta['rows_out']:,} baris x {ta['cols_out']} kolom (0 tanggal <2020 tersisa)", "Berhasil"],
    ["Transform", "Penyimpanan terpisah & isolasi raw", "analytics_* tersimpan terpisah, raw_* tidak tertimpa",
     f"5 tabel di storage: {', '.join(tables_in_db)}", "Berhasil"],
    ["End-to-End", "Eksekusi ELT berurutan (E->L->T)", "Seluruh pipeline berjalan tanpa error & checksum valid",
     f"Berhasil dalam {metrics['pipeline']['elapsed_seconds']}s | checksum={tm['checksum_severity']:,} / {ta['checksum_st_case']:,}", "Berhasil"],
]
test_df = pd.DataFrame(test_rows, columns=["Tahapan", "Aspek yang Diuji", "Hasil yang Diharapkan", "Hasil Aktual", "Status"])
display(test_df)

count_df = pd.DataFrame([
    ["Setelah Extract (Dataset Utama)", em["rows"], "Data mentah awal dari sumber CSV (tanpa cleaning)"],
    ["Setelah Extract (NHTSA Crash API)", ea["rows"], "Data JSON mentah awal dari snapshot NHTSA API"],
    ["Setelah Load Raw (raw_accidents)", lr["counts"]["raw_accidents"], "Raw JSON tersimpan utuh di target storage (skema raw)"],
    ["Setelah Load Raw (raw_nhtsa_api_cases)", lr["counts"]["raw_nhtsa_api_cases"], "Raw JSON tersimpan utuh termasuk 19 baris tidak valid"],
    ["Setelah Transform (analytics_accidents)", tm["rows_out"], "Hasil transformasi SQL di storage (27 kolom analitik)"],
    ["Setelah Transform (analytics_nhtsa_api_cases)", ta["rows_out"], "Hasil transformasi SQL: 11 tanggal sentinel & 8 metrik invalid dibuang"],
    ["Retensi Raw Pasca-Transform (raw_accidents)", rp["raw_accidents"], "Data mentah tetap dipertahankan & terpisah"],
    ["Retensi Raw Pasca-Transform (raw_nhtsa_api_cases)", rp["raw_nhtsa_api_cases"], "Data mentah API tetap utuh (8,798 baris tidak terhapus)"],
], columns=["Tahapan", "Jumlah Baris", "Keterangan"])
display(count_df)

assert em["rows"] == lr["counts"]["raw_accidents"] == rp["raw_accidents"] == tm["rows_out"]
assert ea["rows"] == lr["counts"]["raw_nhtsa_api_cases"] == rp["raw_nhtsa_api_cases"]
assert ea["rows"] - ta["removed_bad_date"] - ta["removed_invalid_metrics"] - ta["removed_duplicate"] == ta["rows_out"]
assert raw_sample_readback == 11 and analytics_sample_readback == 0
assert tm["checksum_severity"] == 576217 and ta["checksum_st_case"] == 2269732984
assert set(tables_in_db) == {
    "analytics_accidents", "analytics_nhtsa_api_cases",
    "raw_accidents", "raw_nhtsa_api_cases", "raw_nhtsa_states",
}
assert all(test_df["Status"] == "Berhasil")
log("TESTING  | Semua pengujian ELT (Extract -> Load Raw -> Transform SQL -> End-to-End) berstatus Berhasil!")
LOG_FILE.write_text("\n".join(LOGS) + "\n", encoding="utf-8")


,Tahapan,Aspek yang Diuji,Hasil yang Diharapkan,Hasil Aktual,Status
0,Extract,Pembacaan sumber utama (raw),"266,656 baris x 21 kolom tanpa cleaning","266,656 baris x 21 kolom (raw str)",Berhasil
1,Extract,Pembacaan NHTSA Crash API (raw),"8,798 kasus JSON + 53 state referensi","8,798 kasus JSON, 53 state",Berhasil
2,Load,Koneksi & struktur skema raw,Tabel raw_* terbentuk dengan kolom JSON,"SQLite supabase_local.db: raw_accidents, raw_n...",Berhasil
3,Load,Penyimpanan raw data utuh,Jumlah raw di storage = hasil Extract,"main: 266,656, api: 8,798, states: 53",Berhasil
4,Load,Pembacaan kembali raw data,Raw JSON (termasuk data kotor) terbaca ulang,11 tanggal sentinel masih utuh di raw_nhtsa_ap...,Berhasil
5,Transform,SQL cleaning & transform (utama),"JSON diekstrak, tipe tanggal, 7 grup cuaca, 27...","266,656 baris x 27 kolom | null=0 | cuaca=7 grup",Berhasil
6,Transform,SQL cleaning & transform (API),"Epoch UTC parsed, CountyName bersih, buang 19 ...","8,798 -> 8,779 baris x 12 kolom (0 tanggal <20...",Berhasil
7,Transform,Penyimpanan terpisah & isolasi raw,"analytics_* tersimpan terpisah, raw_* tidak te...","5 tabel di storage: analytics_accidents, analy...",Berhasil
8,End-to-End,Eksekusi ELT berurutan (E->L->T),Seluruh pipeline berjalan tanpa error & checks...,"Berhasil dalam 46.98s | checksum=576,217 / 2,2...",Berhasil


,Tahapan,Jumlah Baris,Keterangan
0,Setelah Extract (Dataset Utama),266656,Data mentah awal dari sumber CSV (tanpa cleaning)
1,Setelah Extract (NHTSA Crash API),8798,Data JSON mentah awal dari snapshot NHTSA API
2,Setelah Load Raw (raw_accidents),266656,Raw JSON tersimpan utuh di target storage (ske...
3,Setelah Load Raw (raw_nhtsa_api_cases),8798,Raw JSON tersimpan utuh termasuk 19 baris tida...
4,Setelah Transform (analytics_accidents),266656,Hasil transformasi SQL di storage (27 kolom an...
5,Setelah Transform (analytics_nhtsa_api_cases),8779,Hasil transformasi SQL: 11 tanggal sentinel & ...
6,Retensi Raw Pasca-Transform (raw_accidents),266656,Data mentah tetap dipertahankan & terpisah
7,Retensi Raw Pasca-Transform (raw_nhtsa_api_cases),8798,"Data mentah API tetap utuh (8,798 baris tidak ..."


[14:52:54] [INFO] TESTING  | Semua pengujian ELT (Extract -> Load Raw -> Transform SQL -> End-to-End) berstatus Berhasil!


5516

## 6. Analisis dan Kesimpulan

Pipeline **ELT (Extract – Load – Transform)** berhasil diimplementasikan, diintegrasikan, dan divalidasi sesuai
rancangan arsitektur ELT Minggu 9 (*Airbyte → Supabase `raw` → dbt SQL → Supabase `analytics` → Power BI*).

**Perbedaan Utama Implementasi Minggu 10 (ETL) vs Minggu 11 (ELT):**
- **Minggu 10 (ETL):** transformasi dan cleaning dilakukan di memori lokal (`pandas`) **sebelum** data dimuat ke `analytics.*`. Data mentah tidak disimpan di target storage.
- **Minggu 11 (ELT):** seluruh data mentah (`266.656` baris utama, `8.798` kasus JSON API, `53` state referensi) langsung dimuat sebagai JSON ke skema `raw` (`raw_*`) **terlebih dahulu**, lalu dibersihkan dan ditransformasikan **di dalam mesin database** menggunakan SQL (`json_extract`, CTE, window `ROW_NUMBER()`, `JOIN`) menjadi tabel terpisah `analytics_*`, sementara data mentah di `raw_*` tetap dipertahankan utuh (termasuk 11 tanggal sentinel dan 8 baris `Persons=0` yang tersaring di layer `analytics`).
- Kedua pendekatan menghasilkan output analitik yang konsisten identik (`266.656 × 27` kolom dengan checksum `576.217`, serta `8.779 × 12` kolom API dengan checksum `2.269.732.984`).

**Kendala dan Penanganan:**
1. Kredensial database Supabase cloud tidak disimpan di repository; implementasi lokal memakai SQLite 3.53.1 dengan penamaan terpisah `raw_*` dan `analytics_*` di `supabase_local.db`, dilengkapi fungsi `load_raw_to_supabase_pg()` dan skrip `postgres_elt_transform.sql` / `src/etl/elt_supabase.py` untuk eksekusi langsung pada Supabase PostgreSQL.
2. Endpoint NHTSA Crash API memblokir request CLI (`HTTP 403`), sehingga digunakan snapshot hasil pengambilan browser (`data/raw/nhtsa_api_live.json`).
3. Fungsi `INITCAP` tidak tersedia bawaan di SQLite (tersedia bawaan di PostgreSQL); didaftarkan sebagai fungsi SQL deterministik pada koneksi SQLite sehingga query SQL transformasi berjalan 100% di dalam mesin SQLite.
